# Push to GitHub

Puts this folder on GitHub as `github.com/muf148/pqk100`. The data, the 60 MB simulation cache, `application/`,
`junk/` and `.venv/` stay on your computer (see `.gitignore`).

**Before you run it:** go to https://github.com/new while signed in as muf148 and create a repository named
`pqk100`. Choose **Private** and leave it empty: no README, no .gitignore, no license.
Keep it private until the paper is out; then switch it to public under Settings, Danger Zone, Change visibility.

Then run all. The first push opens a GitHub sign-in window. Later, rerun it to push new commits; set `MESSAGE` first.

In [1]:
REMOTE  = "https://github.com/muf148/pqk100.git"
MESSAGE = "PQK100: projected quantum kernels at 156 qubits for genomic classification"
NAME, EMAIL = "Muhammad Faryad", "muhammadfaryad@gmail.com"

In [2]:
import os, sys
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
import shutil, subprocess

class Stop(Exception):
    def _render_traceback_(self):
        return [f"STOPPED: {self}"]

def git(*args, check=True):
    r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if "dubious ownership" in out:
        subprocess.run(["git", "config", "--global", "--add", "safe.directory", ROOT.as_posix()])
        print(f"(told git to trust {ROOT.as_posix()})")
        r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
        out = (r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise Stop(f"git {' '.join(args)} failed:\n{out}")
    return out

if not shutil.which("git"):
    raise Stop("Git is not installed. Get it from https://git-scm.com, then restart VS Code.")
print(git("--version"))

project folder: D:\Active Research Projects\PQK100


Stop: Git is not installed. Get it from https://git-scm.com, then restart VS Code.

## Commit

In [ ]:
if not (ROOT / ".git").exists():
    print(git("init", "-b", "main"))
git("config", "user.name", NAME)
git("config", "user.email", EMAIL)
git("add", ".")
staged = [l for l in git("status", "--short").splitlines() if l.strip()]
print(f"{len(staged)} changed files")
print("\n".join(staged[:120]))
leak = [l for l in staged if any(k in l for k in ("data/", "application/", "junk/", ".venv/", "outputs/exact/"))]
if leak:
    raise Stop("these should not go to GitHub; check .gitignore:\n" + "\n".join(leak))
if staged:
    print(git("commit", "-m", MESSAGE))
else:
    print("Nothing new to commit.")

## Push

In [ ]:
remotes = git("remote")
if "origin" not in remotes.split():
    git("remote", "add", "origin", REMOTE)
else:
    git("remote", "set-url", "origin", REMOTE)
print(git("push", "-u", "origin", "main"))
print("\nDone: " + REMOTE.removesuffix(".git"))

If the push fails with "failed to push some refs", the GitHub repository is not empty. Delete it on GitHub,
create it again without a README, and rerun the Push cell.